# Gmail Assistant Notebook
### Author: Nikhil Gavini (nikhilgavini@gmail.com)
Primarily used to investigate the Gmail API and get it working before moving to 'production' .py files for Data Ingest

## Imports

In [1]:
import os
import os.path
from pathlib import Path
from chromadb import PersistentClient
from chromadb.utils import embedding_functions
from gmail_assistant import config

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

import base64
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders

c:\Users\nikhi\projects\gmail-rag-assistant\.gmail_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\nikhi\projects\gmail-rag-assistant\.gmail_rag\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikhi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to ru

## Google API Service Setup
Follow instructions here: https://developers.google.com/workspace/gmail/api/quickstart/python

Recommend running their quickstart.py first to make sure you initialized everything properly

In [2]:
# If modifying these scopes, delete the file token.json.
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

### Creating the Service as a function

In [3]:
def create_service(client_secret_file, api_name, api_version, scopes):
    CLIENT_SECRET_FILE = client_secret_file
    API_SERVICE_NAME = api_name
    API_VERSION = api_version
    SCOPES = scopes
    
    creds = None
    working_dir = os.getcwd()
    token_path = os.path.join(working_dir, 'token.json')

    '''
    The file token.json stores the user's access and refresh tokens, and is
    created automatically when the authorization flow completes for the first
    time.
    '''
    # If the token exists, use it.
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                CLIENT_SECRET_FILE, SCOPES
            )
        creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        # Call the Gmail API
        service = build(API_SERVICE_NAME, API_VERSION, credentials=creds, static_discovery=False)
        print(API_SERVICE_NAME, API_VERSION, 'service created successfully!')
        return service
    except Exception as e:
        print(e)
        print(f'Failed to create service instance for {API_SERVICE_NAME}.')
        os.remove(token_path)
        return None

### Testing create_service

If this doesn't work and you get a "flow" error, try running google's quickstart.py first to make sure you initialized everything properly

In [4]:
client_secret_file = 'credentials.json'
API_SERVICE_NAME = 'gmail'
API_VERSION = 'v1'

service = create_service(client_secret_file, API_SERVICE_NAME, API_VERSION, SCOPES)

gmail v1 service created successfully!


### Helper Function to initialize a specific gmail service

In [5]:
def init_gmail_service(client_file, api_name='gmail', api_version='v1', scopes=SCOPES):
    return create_service(client_file, api_name, api_version, scopes)

## Fetch Email and Contents from Gmail API

### Extract Body (Private Function)

Helper function to extract body from an email payload

Depending on the email format, the body may be structured differently.

For multi-part emails, the body may be nested within parts of the payload.
- In this case, we search for multipart/alternative
- This contains both plain text and HTML versions of the email

Otherwise, check for plain text only.

We also decode any encoded objeccts using a base64 decoding method to return the body or attachment content

In [6]:
def _extract_body(payload):
    body = '<Text body not available>'
    if 'parts' in payload:
        for part in payload['parts']:
            if part['mimeType'] == 'multipart/alternative':
                for subpart in part['parts']:
                    if subpart['mimeType'] == 'text/plain' and 'data' in subpart['body']:
                        body = base64.urlsafe_b64decode(subpart['body']['data']).decode('utf-8')
                        break
            elif part['mimeType'] == 'text/plain' and 'data' in part['body']:
                body = base64.urlsafe_b64decode(part['body']['data']).decode('utf-8')
                break
    elif 'body' in payload and 'data' in payload['body']:
        body = base64.urlsafe_b64decode(payload['body']['data']).decode('utf-8')
    return body

### Aggregate Email Text


Takes relevant information (subject, sender, message date) and prepends it to the body.

This provides "contextual anchors" for the LLM when it starts chunking.

In [7]:
def _aggregate_email_text(subject, sender, msg_date, body):
    return f"Subject: {subject}\nFrom: {sender}\nDate: {msg_date}\n\n{body}"

### Get List of Folders
Helper function to only return a list of folders that we care about.

In [8]:
def get_list_of_folders(service):
    # Folders we don't care about
    remove_labels = [
      'YELLOW_STAR',
      'TRASH',
      'Notes',
      'CHAT',
      'DRAFT',
      'CATEGORY_PROMOTIONS',
      'CATEGORY_FORUMS',
      'CATEGORY_PERSONAL',
      'CATEGORY_UPDATES',
      'CATEGORY_SOCIAL'
  ]

    results = service.users().labels().list(userId="me").execute()
    labels = results.get("labels", [])

    if not labels:
      print("No labels found.")
      return
    else:
      labels = list({l['name'] for l in labels if 'name' in l})
      good_labels = [x for x in labels if x not in remove_labels]
      return good_labels

In [9]:
get_list_of_folders(service)

['Joint Couple/Vacations/San Diego Trip',
 'Individual/Football',
 'Individual/Learning/Bamboo Weekly',
 'Joint Couple/Housing',
 'Joint Couple/Financial Accounts/Taxes',
 'Individual/Basketball',
 'Individual/Jobs/Boeing: GMD',
 'Joint Couple/Vacations/DC HYROX',
 'Individual',
 'Individual/Learning/Masterclass',
 'STARRED',
 'Individual/Jobs/Booz Allen: MLE/Onboarding',
 'Joint Couple/Vacations',
 'SENT',
 'Individual/Jobs/CACI: USFFC NFL/Onboarding',
 'SPAM',
 'Joint Couple/Residency',
 'Joint Couple/Military',
 'Joint Couple/Vacations/Droese Wedding',
 'Joint Couple/Financial Accounts/Checking',
 'IMPORTANT',
 'Joint Couple/Financial Accounts/Investments',
 'Joint Couple/Vehicles',
 'Individual/Jobs/CACI: USFFC NFL',
 'Joint Couple/Housing/Cleaning',
 'Joint Couple/Financial Accounts',
 'Joint Couple/Recipes',
 'Joint Couple',
 'Joint Couple/Recurring Payments',
 'Joint Couple/Financial Accounts/Savings',
 'Individual/To-Do',
 'Joint Couple/Medical',
 'Joint Couple/Financial Accoun

### Get Email Messages

Gets a list of the email messages using service.users().messages().list()

No details yet, that comes in the next functino

In [10]:
def get_email_messages(service, user_id='me', label_ids=None, folder_name='INBOX', max_results=5):
    messages = []
    next_page_token = None

    if folder_name: # If a folder name is provided, we need to get the label ID for the folder
        label_results = service.users().labels().list(userId=user_id).execute()
        labels = label_results.get('labels', [])
        folder_label_id = next((label['id'] for label in labels if label['name'].lower() == folder_name.lower()), None)
        if folder_label_id:
            if label_ids:
                label_ids.append(folder_label_id)
            else:
                label_ids = [folder_label_id]
        else:
            raise ValueError(f"Folder '{folder_name}' not found.")
    
    while True: # Continue fetching messages until we have reached the max_results or there are no more messages
        result = service.users().messages().list(
            userId = user_id,
            labelIds = label_ids,
            maxResults = min(500, max_results - len(messages)) if max_results else 500, # This method can only fetch 500 messages per API call
            pageToken = next_page_token
        ).execute()

        messages.extend(result.get('messages', []))
        next_page_token = result.get('nextPageToken')

        if not next_page_token or (max_results and len(messages) >= max_results):
            break
    
    return messages[:max_results] if max_results else messages  # Ensures we return exactly the number of messages requested

### Get Email Message Details

In [11]:
def get_email_message_details(service, msg_id):
    message = service.users().messages().get(userId='me', id=msg_id, format='full').execute()
    payload = message['payload']
    headers = payload.get('headers', [])    # Contain important metadata about the email

    subject = next((header['value'] for header in headers if header['name'].lower() == 'subject'), None)
    if not subject:
        subject = message.get('subject', 'No subject')
    
    sender = next((header['value'] for header in headers if header['name'] == 'From'), 'No sender')
    recipients = next((header['value'] for header in headers if header['name'] == 'To'), 'No recipients')
    snippet = message.get('snippet', 'No snippet')
    has_attachments = any(part.get('filename') for part in payload.get('parts', []) if part.get('filename'))
    date = next((header['value'] for header in headers if header['name'] == 'Date'), 'No date')
    star = message.get('labelIds', []).count('STARRED') > 0
    label = ', '.join(message.get('labelIds', []))

    body = _extract_body(payload)

    text = _aggregate_email_text(subject, sender, date, body)

    return {
        'type': 'email',
        'source': subject,
        'text': text,
        'metadata': {
            'id': msg_id,
            'date': date,
            'sender': sender
        }
    }

Let's test it all

In [12]:
service = init_gmail_service(client_secret_file)

gmail v1 service created successfully!


In [13]:
messages = get_email_messages(service, max_results=config.MAX_RESULTS, folder_name='INBOX')

In [14]:
messages

[{'id': '1a04e3283076bfbf', 'threadId': '1a04e3283076bfbf'}]

In [15]:
for msg in messages:
    details = get_email_message_details(service, msg['id'])
    if details:
        print(f"Source: {details['source']}")
        print(f"Text: \n{details['text'][:200]}")
        print(f"Metadata: {details['metadata']}")
        print("-" * 50)

Source: Anusha & Ishaan Are Getting Married!
Text: 
Subject: Anusha & Ishaan Are Getting Married!
From: "Anusha & Ishaan" <anushaishaan2027@gmail.com>
Date: Sat, 29 Aug 2026 11:45:16 -0400

Hello!

We are so excited to officially invite you to celebr
Metadata: {'id': '1a04e3283076bfbf', 'date': 'Sat, 29 Aug 2026 11:45:16 -0400', 'sender': '"Anusha & Ishaan" <anushaishaan2027@gmail.com>'}
--------------------------------------------------


### Fetch Emails Function
Puts everything from this section together

In [16]:
def fetch_emails(service, max_results):
    emails = []
    
    # Loop through all of the folders
    for folder in get_list_of_folders(service):
        print(f"Looking in {folder} folder.")
        messages = get_email_messages(service, max_results=max_results, folder_name=folder)
        # Extract the details and add them to the emails list of dicts
        for msg in messages:
            details = get_email_message_details(service, msg['id'])
            if details:
                details['metadata']['folder'] = folder
                emails.append({
                    'type': 'email',
                    'source': details['source'],
                    'text': details['text'],
                    'metadata': details['metadata']
                })
    
    print(f"Found {len(emails)} emails.")

    return emails

Testing it all

In [17]:
# Initialize the gmail service
service = init_gmail_service(client_secret_file)
list_emails = fetch_emails(service, config.MAX_RESULTS)

gmail v1 service created successfully!
Looking in Joint Couple/Vacations/San Diego Trip folder.
Looking in Individual/Football folder.
Looking in Individual/Learning/Bamboo Weekly folder.
Looking in Joint Couple/Housing folder.
Looking in Joint Couple/Financial Accounts/Taxes folder.
Looking in Individual/Basketball folder.
Looking in Individual/Jobs/Boeing: GMD folder.
Looking in Joint Couple/Vacations/DC HYROX folder.
Looking in Individual folder.
Looking in Individual/Learning/Masterclass folder.
Looking in STARRED folder.
Looking in Individual/Jobs/Booz Allen: MLE/Onboarding folder.
Looking in Joint Couple/Vacations folder.
Looking in SENT folder.
Looking in Individual/Jobs/CACI: USFFC NFL/Onboarding folder.
Looking in SPAM folder.
Looking in Joint Couple/Residency folder.
Looking in Joint Couple/Military folder.
Looking in Joint Couple/Vacations/Droese Wedding folder.
Looking in Joint Couple/Financial Accounts/Checking folder.
Looking in IMPORTANT folder.
Looking in Joint Couple/F

In [18]:
list_emails

[{'type': 'email',
  'source': 'THE EVALUATION - Episode 1: Quarterbacks is Here',
  'text': 'Subject: THE EVALUATION - Episode 1: Quarterbacks is Here\nFrom: "SūmerSports" <marketing@sumersports.com>\nDate: Thu, 12 Feb 2026 22:25:14 +0000\n\nhttps://sumersports.com/?utm_source=SumerSports&utm_campaign=e5d7dda0e0-EMAIL_CAMPAIGN_2025_03_28_09_11_COPY_01&utm_medium=email&utm_term=0_-7512f09cfe-810832463\r\n\r\n\r\n** THE EVALUATION\r\n------------------------------------------------------------\r\n\r\n\r\n** Episode 1: Quarterbacks\r\n------------------------------------------------------------\r\nhttps://youtu.be/G-ojFVIcJRI?si=gcLsi0oN3P8S6xDS&utm_source=SumerSports&utm_campaign=e5d7dda0e0-EMAIL_CAMPAIGN_2025_03_28_09_11_COPY_01&utm_medium=email&utm_term=0_-7512f09cfe-810832463\r\n\r\nNFL scouts Mike Mayock and Mark Ellenz break down the entire 2026 quarterback class, including Heisman winner Fernando Mendoza, Carson Beck, Ty Simpson, and more. Plus, two-time Super Bowl champion Eli Ma

## Explore the Vector DB after we ran ingest.py

In [19]:
chroma = PersistentClient(path=config.DB_PATH)
collection = chroma.get_or_create_collection(
    name=config.COLLECTION_NAME, 
    embedding_function=config.EMBEDDING_FUNCTION
)

# Assuming 'collection' is already defined in answer.py
results = collection.get(include=["metadatas"])
all_metadatas = results["metadatas"]

# Use a set to get unique source names
unique_sources = list(set(m['source'] for m in all_metadatas if 'source' in m))

print(f"Found {len(unique_sources)} unique documents:")
for source in unique_sources:
    print(f"- {source}")

Found 7 unique documents:
- Start your year with 40% off Headspace
- You haven't showered since last year??
- Breaking news: U.S. will “run” Venezuela after Maduro’s capture, Trump says
- Don't lose your 19-day Streak!
- New year wishes from Airlearn ✨
- 3 days into 2026... still no resolutions planned?
- Nikhil, your Streak Protect is running low!
